In [2]:
import tensorflow as tf
import numpy as np

In [3]:
template_training = tf.keras.utils.image_dataset_from_directory(
    'dataset/training_set',
    image_size=(64, 64),
    batch_size=32,
    label_mode='binary',
    shuffle=True,
    seed=1,
)

train_datagen = tf.keras.Sequential([
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomFlip('horizontal'),
])

print('Class names:', template_training.class_names)
print('Class indices:', {'cats': 0, 'dogs': 1})
print('Batches:', tf.data.experimental.cardinality(template_training).numpy())

Found 8000 files belonging to 2 classes.
Class names: ['cats', 'dogs']
Class indices: {'cats': 0, 'dogs': 1}
Batches: 250


In [4]:
template_test = tf.keras.utils.image_dataset_from_directory(
    'dataset/test_set',
    image_size=(64, 64),
    batch_size=32,
    label_mode='binary',
    shuffle=False,
)

print('Class names:', template_test.class_names)
print('Batches:', tf.data.experimental.cardinality(template_test).numpy())

Found 2000 files belonging to 2 classes.
Class names: ['cats', 'dogs']
Batches: 63


In [5]:
cnn = tf.keras.models.Sequential()

In [6]:
cnn.add(tf.keras.layers.Rescaling(1./255, input_shape=(64, 64, 3)))
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))

C:\Users\andhi\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

In [8]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

In [9]:
cnn.add(tf.keras.layers.Flatten())

In [10]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

In [11]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [12]:
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [13]:
training_set = template_training.map(
    lambda x, y: (train_datagen(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE,
).prefetch(tf.data.AUTOTUNE)

test_set = template_test.prefetch(tf.data.AUTOTUNE)

cnn.fit(x=training_set, validation_data=test_set, epochs=25)

Epoch 1/25


C:\Users\andhi\AppData\Roaming\Python\Python314\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.5794 - loss: 0.6752 - val_accuracy: 0.6635 - val_loss: 0.6307
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.6580 - loss: 0.6183 - val_accuracy: 0.7140 - val_loss: 0.5698
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7046 - loss: 0.5696 - val_accuracy: 0.7315 - val_loss: 0.5410
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7306 - loss: 0.5319 - val_accuracy: 0.7270 - val_loss: 0.5458
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7511 - loss: 0.5036 - val_accuracy: 0.7675 - val_loss: 0.4988
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7722 - loss: 0.4783 - val_accuracy: 0.7575 - val_loss: 0.5216
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7797 - loss: 0.4598 - val_accuracy: 0.7755 - val_loss: 0.4815
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7859 - loss: 0.4420 - val_accuracy: 0.767

In [14]:
test_image = tf.keras.utils.load_img(
    'dataset/single_prediction/cat_or_dog_1.jpg',
    target_size=(64, 64),
)
test_image = tf.keras.utils.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis=0)

result = cnn.predict(test_image)

print('Class indices:', {'cats': 0, 'dogs': 1})
print('Raw output:', float(result[0][0]))

if result[0][0] > 0.5:
    prediction = 'dogs'
else:
    prediction = 'cats'

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Class indices: {'cats': 0, 'dogs': 1}
Raw output: 0.9999995231628418
dogs


In [15]:
test_image_2 = tf.keras.utils.load_img(
    'dataset/single_prediction/cat_or_dog_2.jpg',
    target_size=(64, 64),
)
test_image_2 = tf.keras.utils.img_to_array(test_image_2)
test_image_2 = np.expand_dims(test_image_2, axis=0)

result_2 = cnn.predict(test_image_2)

print('Raw output:', float(result_2[0][0]))
print('dogs' if result_2[0][0] > 0.5 else 'cats')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Raw output: 0.07591787725687027
cats
